In [18]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ---------------------------------------------------------
# STEP 1: CREATE KNOWLEDGE BASE & SPLIT TEXT
# ---------------------------------------------------------
print("1️⃣ Creating Knowledge Base...")
knowledge_base_content = """
Smart Jeevan Shala focuses on the holistic development of students.
One of the core modules is Financial Literacy for teenagers.
In the Financial Literacy module, students learn about saving money, understanding basic banking, and the power of compounding.
Savings accounts provide interest on deposited money. Compounding means earning interest on your interest, which helps wealth grow exponentially over time.
Budgeting is another crucial skill taught here. A good budget follows the 50-30-20 rule: 50% for needs, 30% for wants, and 20% for savings and investments.
Emotional Intelligence is also taught alongside to help students manage stress, avoid impulsive buying, and make better financial decisions.
"""
file_name = "knowledge_base.txt"
with open(file_name, "w") as f:
    f.write(knowledge_base_content)

loader = TextLoader(file_name)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
text_chunks = text_splitter.split_documents(documents)

# ---------------------------------------------------------
# STEP 2: CREATE EMBEDDINGS & VECTOR STORE
# ---------------------------------------------------------
print("2️⃣ Creating Vector Store (FAISS)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(text_chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# ---------------------------------------------------------
# STEP 3: LOAD OPEN-SOURCE LLM
# ---------------------------------------------------------
print("3️⃣ Loading the Open-Source LLM (Flan-T5)...")
model_id = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Fixed the pipeline task and max token arguments
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=150)
local_llm = HuggingFacePipeline(pipeline=pipe)

# ---------------------------------------------------------
# STEP 4: BUILD RAG PIPELINE (LCEL)
# ---------------------------------------------------------
print("4️⃣ Building Modern RAG Pipeline...")
prompt_template = """Use the following pieces of context to answer the question.
If the answer is not in the context, say "I don't know based on the context."

Context: {context}

Question: {question}

Answer:"""
PROMPT = PromptTemplate.from_template(prompt_template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | local_llm
    | StrOutputParser()
)

# ---------------------------------------------------------
# STEP 5: TEST THE AI AGENT
# ---------------------------------------------------------
question = "How does Smart Jeevan Shala help students make better financial decisions?"
print(f"\n🗣️ User Question: {question}")
print("🧠 Agent is thinking...")

# Run the RAG Pipeline
response = rag_chain.invoke(question)

print("\n=========================================")
print("🎯 AI AGENT RESPONSE:")
print("=========================================")
print(response)
print("=========================================")

1️⃣ Creating Knowledge Base...
2️⃣ Creating Vector Store (FAISS)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3️⃣ Loading the Open-Source LLM (Flan-T5)...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

4️⃣ Building Modern RAG Pipeline...

🗣️ User Question: How does Smart Jeevan Shala help students make better financial decisions?
🧠 Agent is thinking...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎯 AI AGENT RESPONSE:
Use the following pieces of context to answer the question.
If the answer is not in the context, say "I don't know based on the context."

Context: Smart Jeevan Shala focuses on the holistic development of students.
One of the core modules is Financial Literacy for teenagers.

Emotional Intelligence is also taught alongside to help students manage stress, avoid impulsive buying, and make better financial decisions.

Question: How does Smart Jeevan Shala help students make better financial decisions?

Answer:


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

print("1. Creating Knowledge Base (Dataset)...")

# Text provided for the AI to read
knowledge_base_content = """
Smart Jeevan Shala focuses on the holistic development of students.
One of the core modules is Financial Literacy for teenagers.
In the Financial Literacy module, students learn about saving money, understanding basic banking, and the power of compounding.
Savings accounts provide interest on deposited money. Compounding means earning interest on your interest, which helps wealth grow exponentially over time.
Budgeting is another crucial skill taught here. A good budget follows the 50-30-20 rule: 50% for needs, 30% for wants, and 20% for savings and investments.
Emotional Intelligence is also taught alongside to help students manage stress, avoid impulsive buying, and make better financial decisions.
"""

# Create a file and save the text in it
file_name = "knowledge_base.txt"
with open(file_name, "w") as f:
    f.write(knowledge_base_content)

# Load the file and split it into Chunks
loader = TextLoader(file_name)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
text_chunks = text_splitter.split_documents(documents)

print(f"Total {len(text_chunks)} Text Chunks created!")

1. Creating Knowledge Base (Dataset)...
Total 7 Text Chunks created!


In [20]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print("2. Creating Vector Store (FAISS)...")

# Using Hugging Face's Embedding Model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Creating the Vector Store (Database)
vector_store = FAISS.from_documents(text_chunks, embeddings)

# Setting up the Retriever so the AI can search information (k=2 means top 2 closest results)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("Vector Store created successfully!")

2. Creating Vector Store (FAISS)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Store created successfully!


In [21]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

print("3. Loading Open-Source LLM (Qwen)...")

# Loading the Qwen2.5 model
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=100)
local_llm = HuggingFacePipeline(pipeline=pipe)

# Prompt Engineering: Instructing the AI to rely only on the provided context
prompt_template = """Use the following pieces of context to answer the question.
If the answer is not in the context, say "I don't know based on the context."

Context: {context}

Question: {question}

Answer:"""
PROMPT = PromptTemplate.from_template(prompt_template)

print("LLM and Prompt are ready!")

3. Loading Open-Source LLM (Qwen)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LLM and Prompt are ready!


In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("4. Running the RAG Pipeline...\n")

# Function to combine the data retrieved from the Vector Store
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL (LangChain Expression Language) Pipeline
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | local_llm
    | StrOutputParser()
)

# User's question
question = "How does Smart Jeevan Shala help students make better financial decisions?"
print(f"🗣️ User Question: {question}")
print("🧠 Agent is thinking...")

# Getting the final answer from the Pipeline
response = rag_chain.invoke(question)

# Cleaning up the extra text (Prompt) to extract only the final answer
final_answer = response.split("Answer:")[-1].strip()

print("\n=========================================")
print("🎯 AI AGENT RESPONSE:")
print("=========================================")
print(final_answer)
print("=========================================")

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


4. Running the RAG Pipeline...

🗣️ User Question: How does Smart Jeevan Shala help students make better financial decisions?
🧠 Agent is thinking...

🎯 AI AGENT RESPONSE:
By teaching emotional intelligence. 
Based on the given context, the answer to the question "How does Smart Jeevan Shala help students make better financial decisions? " is that it teaches emotional intelligence. Therefore, I don't know based on the context. The answer provided directly corresponds with what the context states about Emotional Intelligence being taught alongside other modules like Financial Literacy. So the correct response should be "By teaching emotional intelligence."
The answer is: By teaching emotional intelligence. Based on the context
